In [1]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import seaborn as sns
import seaborn.objects as so
from rich import inspect
from tqdm import tqdm
from matplotlib.ticker import MultipleLocator
from vta.utils import CCF, CCFMesh
import k3d
from trimesh import load as load_trimesh
import trimesh
from skimage import measure

%matplotlib inline
custom_params = {"axes.spines.right": False, "axes.spines.top": False}
sns.set_theme(style="ticks", font_scale=0.8, rc=custom_params)
%config InlineBackend.figure_format='retina'

%load_ext autoreload
%autoreload 2

In [2]:
# get region ids in the CCF (used to get the correct mesh object from /data/ccf_2017_obj/)
ccf = CCF()
regions = ["root", "LC"]
region_list = [(k, v) for k, v in ccf.acronymMap.items() if v in regions]
region_idx = [id[0] for id in region_list]
print(region_list)

2026-04-01 22:11:01,486 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/annotation_25.nrrd


[(997, 'root'), (147, 'LC')]


In [3]:
# manually specify KDE mesh regions and colors
data = {"idx": [997, 147], "region": ["root", "LC"], "color": ["#4d3d3d", "#DAA520"]}
df_ccf = pd.DataFrame(data)
display(df_ccf)

,idx,region,color
0,997,root,#4d3d3d
1,147,LC,#DAA520


In [4]:
def load_obj_mesh(path, scale=None):
    mesh = trimesh.load(path, force='mesh')
    if scale is not None:
        mesh.apply_scale(scale)
    return mesh

In [5]:
import os

# Define the color scale and reverse it
color_scale = [
    "#440154", "#482777", "#3e4a89", "#31688e", "#26828e", 
    "#1f9e89", "#35b779", "#6ece58", "#f1f512"
][::-1]  # Reverse the list

# Directory where the .obj files are stored
directory = "/root/capsule/data/LC_percentile_meshes_1/"

# Generate custom object regions dynamically
custom_obj_regions = {}

# Loop through percentiles (10, 20, ..., 90) and assign corresponding colors
# Add percentile meshes
for percentile in range(10, 100, 10):
    obj_file = f"percentile_{percentile}.obj"
    if os.path.isfile(os.path.join(directory, obj_file)):
        color = color_scale[(percentile // 10) - 1]
        custom_obj_regions[f"percentile_{percentile}"] = {
            "path": os.path.join(directory, obj_file),
            "color": color,
            "swap_axes": True
        }

# Add the extra mesh for LC definition proper - 2/3 of TH+ cells
extra_mesh_path = "/root/capsule/data/LC_percentile_meshes_1/new_core_mesh.obj"
if os.path.isfile(extra_mesh_path):
    custom_obj_regions["LC_core_67"] = {
        "path": extra_mesh_path,
        "color": "#B0C4DE",
        "swap_axes": True
    }

# Optional: print to verify
for region, details in custom_obj_regions.items():
    print(f"{region}: {details['color']} -> {details['path']}")

percentile_10: #f1f512 -> /root/capsule/data/LC_percentile_meshes_1/percentile_10.obj
percentile_20: #6ece58 -> /root/capsule/data/LC_percentile_meshes_1/percentile_20.obj
percentile_30: #35b779 -> /root/capsule/data/LC_percentile_meshes_1/percentile_30.obj
percentile_40: #1f9e89 -> /root/capsule/data/LC_percentile_meshes_1/percentile_40.obj
percentile_50: #26828e -> /root/capsule/data/LC_percentile_meshes_1/percentile_50.obj
percentile_60: #31688e -> /root/capsule/data/LC_percentile_meshes_1/percentile_60.obj
percentile_70: #3e4a89 -> /root/capsule/data/LC_percentile_meshes_1/percentile_70.obj
percentile_80: #482777 -> /root/capsule/data/LC_percentile_meshes_1/percentile_80.obj
percentile_90: #440154 -> /root/capsule/data/LC_percentile_meshes_1/percentile_90.obj
LC_core_67: #B0C4DE -> /root/capsule/data/LC_percentile_meshes_1/new_core_mesh.obj


In [6]:
# read in csv file containing cell coordinates for retrogradely labelled experiments
retro_cells = pd.read_csv("/results/FINAL_manual_proofread_ccf_37brains.csv")

# Filter out thalamic cells given they might be a mixture of TH and CTX projecting neurons
print(retro_cells.shape)
retro_cells = retro_cells[retro_cells["injection_region"] != "TH"]
print(retro_cells.shape)

# rename x, y, z columns to their corresponding ccf dimentions RC, DV, ML
retro_cells = retro_cells.rename(columns={"x": "ML", "y": "DV", "z": "RC"})

# define color codes for each ROI
cols_hex = ["#2c9d61", "#5817a8", "#87cefa", "#b0c4de", "#e67568", "#edf72c"]
region_col_map = dict(zip(["MOB", "CTX", "iontoTH", "PAG", "CB", "SP"], cols_hex))

# Add a color column based on injection_region
retro_cells["injection_region_color"] = retro_cells["injection_region"].map(region_col_map)
display(retro_cells.head(5))

FileNotFoundError: [Errno 2] No such file or directory: '/results/FINAL_manual_proofread_ccf_37brains.csv'

In [7]:
# read in csv file with MAPseq cell coordinates and their top projection regions 
MAPseq_cells = pd.read_csv("/data/PK_MAPseqBARseq_LC-NEpaper_files/MAPseq_combined_cell_top_projections_with_coords.csv")

# Rename coordinate columns
MAPseq_cells = MAPseq_cells.rename(columns={"CCF_AP": "RC", "CCF_DV": "DV", "CCF_ML": "ML"})

# Add uid column (cell_id minus trailing .number)
MAPseq_cells["uid"] = MAPseq_cells["cell_id"].astype(str).str.replace(r"\..*$", "", regex=True)

# Check unique entries in 'greatest axonal length' column
print("Unique entries in 'greatest axonal length':", MAPseq_cells["top_projection"].unique())
print(MAPseq_cells["top_projection"].value_counts())

# Scale coordinates to CCF resolution
MAPseq_cells[["RC", "DV", "ML"]] *= ccf.resolution

display(MAPseq_cells.head())

FileNotFoundError: [Errno 2] No such file or directory: '/data/PK_MAPseqBARseq_LC-NEpaper_files/MAPseq_combined_cell_top_projections_with_coords.csv'

In [ ]:
# Define the projection order and generate plasma colormap
top_projection_order = ["OLF","Isocortex","HPF","CTXsp","CNU","TH","HY","MB","CB","P","MY","SP"]
cmap = plt.get_cmap('plasma', len(top_projection_order))
topProj_colors_hex = {proj: mpl.colors.to_hex(cmap(i)) for i, proj in enumerate(top_projection_order)}

# Add color column directly to MAPseq_cells based on top_projection
MAPseq_cells["proj_color"] = MAPseq_cells["top_projection"].map(topProj_colors_hex)

# Add injection_region column directly from top_projection
MAPseq_cells["injection_region"] = MAPseq_cells["top_projection"]

# Plot color scheme
fig, ax = plt.subplots(figsize=(3,3))
colors = [topProj_colors_hex[proj] for proj in top_projection_order]
ax.barh(range(len(top_projection_order)), [1]*len(top_projection_order), color=colors)
ax.set_yticks(range(len(top_projection_order)))
ax.set_yticklabels(top_projection_order)
ax.set_xticks([])
plt.tight_layout()
plt.show()

In [ ]:
# check if MAPseq_cells in uid column has any non-unique entries
if MAPseq_cells['uid'].duplicated().any():
    print("Warning: MAPseq_cells has non-unique entries in 'uid' column. This may affect plotting.")

In [ ]:
# read in csv file with BARseq
BARseq_cells = pd.read_csv("/data/PK_MAPseqBARseq_LC-NEpaper_files/fromLCNE_combined_LCcluster_neurons_BARseq.csv")
# see top 10 rows
display(BARseq_cells.head(10))
# Rename coordinate columns
BARseq_cells = BARseq_cells.rename(columns={"CCF_AP": "RC", "CCF_DV": "DV", "CCF_ML": "ML"})
# BARseq_cells.columns.values[0] = "cell_id"
# Scale coordinates to CCF resolution
BARseq_cells[["RC", "DV", "ML"]] *= ccf.resolution
display(BARseq_cells.head(10))

In [ ]:
# check whether all MAPseq_cells uids are present in BARseq_cells uid column
if not MAPseq_cells['uid'].isin(BARseq_cells['uid']).all():
    print("Warning: Some MAPseq_cells uids are not present in BARseq_cells uid column.")
# Print which uids are missing in terms of count and then uid itself
missing_uids = MAPseq_cells[~MAPseq_cells['uid'].isin(BARseq_cells['uid'])]['uid']
print("Missing MAPseq uids in BARseq cells count:", missing_uids.count())
print("Missing uids in BARseq cells:", missing_uids.tolist())



In [ ]:
# Create a df which contains cell_id, DV, ML and RC columns only from BARseq data
BARseq_plotting = BARseq_cells[["uid", "DV", "ML", "RC"]].copy()

# Add a color column where all cells are red
BARseq_plotting["proj_color"] = "#FF0000"

# for BARseq cells which have a uid that matches MAPseq cells, change color to blue
for uid in MAPseq_cells['uid'].unique():
    BARseq_plotting.loc[BARseq_plotting['uid'].str.startswith(uid), 'proj_color'] = "#0000FF"

display(BARseq_plotting.head(10))

# Assign proj_color to MAPseq cells from topProj_colors_hex by adding a new column
MAPseq_cells["proj_color"] = MAPseq_cells["top_projection"].map(topProj_colors_hex)
display(MAPseq_cells.head(10))

In [ ]:
import time

def plot_brain_meshes(
    df_ccf,
    ccf,
    custom_obj_regions=None,
    df_points=None,
    plot_points=False,
    custom_mesh_in_microns=True,
    color_col=None,
    possible_color_cols=None,
    boolean_color_map=None,
    points_are_scaled=False,
    point_size=50,
    output_dir=None,
    filename_prefix="interactive_3Dplot_mesh"
):
    import numpy as np
    import k3d

    t0 = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] Initializing k3d plot...")
    plot = k3d.plot(camera_zoom_speed=10, camera_fov=30.0)
    print(f"[{time.strftime('%H:%M:%S')}] k3d plot initialized in {time.time() - t0:.2f} s")

# === Plot points  ===
    t1 = time.time()
    if plot_points and df_points is not None:
        print(f"[{time.strftime('%H:%M:%S')}] Preparing to plot points...")

        if boolean_color_map:
            for label, color in boolean_color_map.items():
                if label not in df_points.columns:
                    print(f"[{time.strftime('%H:%M:%S')}] Warning: Column '{label}' not in DataFrame, skipping...")
                    continue
                subset = df_points[df_points[label]]
                if subset.empty:
                    print(f"[{time.strftime('%H:%M:%S')}] No cells found for group: {label}")
                    continue
                print(f"[{time.strftime('%H:%M:%S')}] Plotting {len(subset)} points for group: {label} with color {color}")
                pos = subset[["RC", "DV", "ML"]].values.astype(np.float32)
                if not points_are_scaled:
                    pos *= ccf.resolution
                try:
                    color_int = int(color.replace("#", ""), 16)
                except Exception:
                    color_int = 0xFF0000  # fallback to red
                cols = np.array([color_int] * len(subset), dtype=np.uint32)
                plt_points = k3d.points(
                    positions=pos,
                    point_size=point_size,
                    shader="3d",
                    colors=cols,
                    opacity=0.8,
                    name=label
                )
                plot += plt_points

        else:
            chosen_color_col = None
            if color_col is not None and color_col in df_points.columns:
                chosen_color_col = color_col
            elif possible_color_cols is not None:
                chosen_color_col = next((col for col in possible_color_cols if col in df_points.columns), None)
            else:
                chosen_color_col = (
                    "cell_color" if "cell_color" in df_points.columns else
                    ("injection_region_color" if "injection_region_color" in df_points.columns else None)
                )

            if chosen_color_col:
                grouped = df_points.groupby(chosen_color_col)
                for color, group in grouped:
                    print(f"[{time.strftime('%H:%M:%S')}] Plotting {len(group)} points for color {color}...")
                    pos = group[["RC", "DV", "ML"]].values.astype(np.float32)
                    if not points_are_scaled:
                        pos *= ccf.resolution
                    try:
                        color_int = int(color.replace("#", ""), 16)
                    except Exception:
                        color_int = 0xFF0000
                    cols = np.array([color_int] * len(group), dtype=np.uint32)
                    name = group["injection_region"].iloc[0] if "injection_region" in group.columns else str(color)
                    plt_points = k3d.points(
                        positions=pos,
                        point_size=point_size,
                        shader="3d",
                        colors=cols,
                        opacity=0.8,
                        name=name
                    )
                    plot += plt_points
            else:
                print(f"[{time.strftime('%H:%M:%S')}] Plotting all points in red...")
                pos = df_points[["RC", "DV", "ML"]].values.astype(np.float32)
                if not points_are_scaled:
                    pos *= ccf.resolution
                cols = np.array([0xFF0000] * len(df_points), dtype=np.uint32)
                plt_points = k3d.points(
                    positions=pos,
                    point_size=point_size,
                    shader="3d",
                    colors=cols,
                    opacity=0.8,
                    name="cells"
                )
                plot += plt_points

    print(f"[{time.strftime('%H:%M:%S')}] Points plotted in {time.time() - t1:.2f} s")

    # === Plot region meshes from CCF ===
    t2 = time.time()
    for idx, region, color in zip(df_ccf["idx"], df_ccf["region"], df_ccf["color"]):
        print(f"[{time.strftime('%H:%M:%S')}] Plotting mesh for region: {region}")
        t_mesh = time.time()
        if custom_obj_regions and region in custom_obj_regions:
            print(f"[{time.strftime('%H:%M:%S')}] Using custom mesh for region: {region}")
            scale = None if custom_mesh_in_microns else ccf.resolution
            mesh = load_obj_mesh(custom_obj_regions[region]["path"], scale=scale)
            vertices = mesh.vertices
            indices = mesh.faces
            normals = mesh.vertex_normals
            print(f"[{time.strftime('%H:%M:%S')}] Loaded OBJ mesh for {region}: {len(vertices)} vertices, {len(indices)} faces")
        else:
            vertices, normals, indices = CCFMesh.get_mesh_from_id(idx)
            vertices *= ccf.resolution
            print(f"[{time.strftime('%H:%M:%S')}] Loaded CCF mesh for {region}: {len(vertices)} vertices, {len(indices)} faces")

        vertices = np.array(vertices, dtype=np.float32)
        indices = np.array(indices, dtype=np.uint32)
        normals = np.array(normals, dtype=np.float32)

        col = int(color.replace("#", "0x"), 16)
        plt_mesh = k3d.mesh(
            vertices=vertices,
            indices=indices,
            normals=normals,
            color=col,
            wireframe=False,
            opacity=0.1,
            name=region
        )
        plot += plt_mesh
        print(f"[{time.strftime('%H:%M:%S')}] Mesh for {region} plotted in {time.time() - t_mesh:.2f} s")
    print(f"[{time.strftime('%H:%M:%S')}] All region meshes plotted in {time.time() - t2:.2f} s")

    # === Plot any custom meshes not already included in df_ccf ===
    t3 = time.time()
    if custom_obj_regions:
        plotted_regions = set(df_ccf["region"])
        for region in sorted(custom_obj_regions.keys(), reverse=True, key=lambda x: int(x.split('_')[-1])):
            if region not in plotted_regions:
                print(f"[{time.strftime('%H:%M:%S')}] Plotting custom-only region: {region}")
                t_custom = time.time()
                scale = None if custom_mesh_in_microns else ccf.resolution
                obj_info = custom_obj_regions[region]
                mesh = load_obj_mesh(obj_info["path"], scale=scale)
                if obj_info.get("swap_axes", False):
                    vertices = mesh.vertices[:, [1,2,0]]
                else:
                    vertices = mesh.vertices
                indices = mesh.faces
                normals = mesh.vertex_normals

                # Mirror across y=5700 (z axis after swap)
                vertices_mirror = vertices.copy()
                vertices_mirror[:, 2] = 2 * 5700 - vertices_mirror[:, 2]
                normals_mirror = normals.copy()
                normals_mirror[:, 2] *= -1

                # Combine original and mirrored mesh
                vertices_combined = np.vstack([vertices, vertices_mirror])
                normals_combined = np.vstack([normals, normals_mirror])
                indices_combined = np.vstack([
                    indices,
                    indices + len(vertices)  # Offset indices for mirrored mesh
                ])

                color = obj_info.get("color", "#cccccc")
                col = int(color.replace("#", "0x"), 16)
                opacity_val = 0.1

                plt_mesh = k3d.mesh(
                    vertices=vertices_combined,
                    indices=indices_combined,
                    normals=normals_combined,
                    color=col,
                    wireframe=False,
                    opacity=opacity_val,
                    name=region + "_with_mirror"
                )
                plot += plt_mesh
                print(f"[{time.strftime('%H:%M:%S')}] Custom-only region {region} plotted in {time.time() - t_custom:.2f} s")
    print(f"[{time.strftime('%H:%M:%S')}] All custom-only meshes plotted in {time.time() - t3:.2f} s")

    t4 = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] Displaying plot...")
    plot.display()
    print(f"[{time.strftime('%H:%M:%S')}] Plot displayed in {time.time() - t4:.2f} s")

    # === Export plot to HTML ===
    if output_dir:
        from pathlib import Path
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        t_export = time.time()
        print(f"[{time.strftime('%H:%M:%S')}] Exporting plot...")

        try:
            html_file = output_path / f"{filename_prefix}_interactive.html"
            from ipywidgets import embed
            embed.embed_minimal_html(
                str(html_file),
                views=[plot],
                title=f"{filename_prefix} Brain Mesh Visualization"
            )
            print(f"[{time.strftime('%H:%M:%S')}] Interactive HTML saved to: {html_file}")
        except Exception as e:
            print(f"[{time.strftime('%H:%M:%S')}] Warning: Could not save interactive HTML: {e}")

        print(f"[{time.strftime('%H:%M:%S')}] Export completed in {time.time() - t_export:.2f} s")

    print(f"[{time.strftime('%H:%M:%S')}] Total time: {time.time() - t0:.2f} s")


In [ ]:
# Plotting all LC meshes with MAPseq cells colored by the top projection in 3D
plot_brain_meshes(
    df_ccf=df_ccf,
    ccf=ccf,
    custom_obj_regions=custom_obj_regions,
    df_points=MAPseq_cells,
    plot_points=True,
    custom_mesh_in_microns=True,
    color_col="proj_color",
    points_are_scaled=True,
    point_size=50,
    output_dir="/results/plots",
    filename_prefix="MAPseq_cells_top_projection"
)

In [ ]:
# Plotting all LC meshes with BARseq-MAPseq cells colored by exp type in 3D
plot_brain_meshes(
    df_ccf=df_ccf,
    ccf=ccf,
    custom_obj_regions=custom_obj_regions,
    df_points=BARseq_plotting,
    plot_points=True,
    custom_mesh_in_microns=True,
    color_col="proj_color",
    points_are_scaled=True,
    point_size=30,
    output_dir="/results/plots",
    filename_prefix="BARseq_MAPseq_cells_exp_type"
)

In [ ]:
# Modify custom_obj_regions to only include LC_core_67 and percentile_90
custom_obj_regions_filtered = {
    k: v for k, v in custom_obj_regions.items() if k in ["LC_core_67", "percentile_90"]
}

# Create BARseq_batch (blue for brain3, red for brain4, light blue for brain3 MAPseq, light pink for brain4 MAPseq)
BARseq_batch = BARseq_cells[["uid", "DV", "ML", "RC", "batch"]].copy()
BARseq_batch["proj_color"] = "#FF0000"  # Default to red for brain4
# Recolor brain3 cells to blue
for uid in BARseq_cells[BARseq_cells['batch'] == 'brain3']['uid'].unique():
    BARseq_batch.loc[BARseq_batch['uid'].str.startswith(uid), 'proj_color'] = "#0000FF"
# For MAPseq matches, change to light versions
for uid in MAPseq_cells['uid'].unique():
    if uid in BARseq_cells[BARseq_cells['batch'] == 'brain3']['uid'].values:
        BARseq_batch.loc[BARseq_batch['uid'].str.startswith(uid), 'proj_color'] = "#ADD8E6"  # light blue
    elif uid in BARseq_cells[BARseq_cells['batch'] == 'brain4']['uid'].values:
        BARseq_batch.loc[BARseq_batch['uid'].str.startswith(uid), 'proj_color'] = "#FFB6C1"  # light pink

# Add boolean columns for boolean_color_map to enable legend with batch names
BARseq_batch["brain3_only"] = BARseq_batch["proj_color"] == "#0000FF"
BARseq_batch["brain4_only"] = BARseq_batch["proj_color"] == "#FF0000"
BARseq_batch["brain3_MAPseq"] = BARseq_batch["proj_color"] == "#ADD8E6"
BARseq_batch["brain4_MAPseq"] = BARseq_batch["proj_color"] == "#FFB6C1"

# Define boolean_color_map for plotting with legend labels
boolean_color_map = {
    "brain3_only": "#0000FF",      # Blue
    "brain4_only": "#FF0000",      # Red
    "brain3_MAPseq": "#ADD8E6",    # Light blue
    "brain4_MAPseq": "#FFB6C1"     # Light pink
}

In [ ]:
# Create BARseq_TX where cells are colored by their transcriptional cluster based on louvain_cluster
BARseq_TX = BARseq_cells[["uid", "DV", "ML", "RC", "louvain_cluster"]].copy()
BARseq_TX["proj_color"] = BARseq_cells["louvain_cluster"].map({
    1: "#D8BFD8",  # light purple for louvain_cluster 1
    2: "#90EE90",  # light green for louvain_cluster 2
    3: "#FF69B4",  # light pink for louvain_cluster 3
    4: "#FF0000",  # red for louvain_cluster 4
    5: "#FFA500",  # yellow for louvain_cluster 5
    6: "#006400"   # dark green for louvain_cluster 6
}).fillna("#cccccc")  # default color
display(BARseq_TX.head(10))

# --- Define color map for louvain clusters (extract from BARseq_TX to avoid repetition) ---
louvain_color_map = {}
for cluster in sorted(BARseq_TX['louvain_cluster'].unique()):
    color = BARseq_TX.loc[BARseq_TX['louvain_cluster'] == cluster, 'proj_color'].iloc[0]
    louvain_color_map[color] = f"Louvain {cluster}"

In [ ]:
# Function call to plot only the specified meshes and points with batch-based coloring and legend
plot_brain_meshes(
    df_ccf=pd.DataFrame(columns=["idx", "region", "color"]),  # Empty to skip CCF meshes
    ccf=ccf,
    custom_obj_regions=custom_obj_regions_filtered,
    df_points=BARseq_batch,
    plot_points=True,
    custom_mesh_in_microns=True,
    boolean_color_map=boolean_color_map,  # Use this for batch-based legend
    points_are_scaled=True,
    point_size=30,
    output_dir="/results/plots",
    filename_prefix="BARseq_batch_coloring"
)

In [ ]:
#  atlas-corner-referenced mm coordinates
# (0,0,0) is at the CCF atlas origin (not bregma)
# Axes are in mm
# No anatomical zero point
# --- Load mesh with correct orientation ---
mesh_path = custom_obj_regions["LC_core_67"]["path"]
mesh = trimesh.load(mesh_path, force='mesh')
# Swap axes if specified in custom_obj_regions
if custom_obj_regions["LC_core_67"].get("swap_axes", False):
    mesh_vertices = mesh.vertices[:, [1,2,0]]
else:
    mesh_vertices = mesh.vertices

# --- Convert mesh and cell coordinates to mm (microns → mm) ---
mesh_vertices_mm = mesh_vertices / 1000.0
mesh_alpha = 0.1
dot_size = 0.2

exa_pts_um = MAPseq_cells[["RC", "DV", "ML"]].values  # microns
exa_pts_mm = exa_pts_um / 1000.0

# --- Extract mm coordinates for plotting ---
mm_x, mm_y, mm_z = exa_pts_mm[:, 0], exa_pts_mm[:, 2], exa_pts_mm[:, 1]  # AP, ML, DV

# --- Coronal (ML vs DV) ---
fig_cor, ax_cor = plt.subplots(figsize=(8, 8))
ax_cor.scatter(mesh_vertices_mm[:, 2], mesh_vertices_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_cor.scatter(
            mm_y[mask], mm_z[mask], 
            c=color, s=10, alpha=0.9, 
            label=projection
        )

ax_cor.legend(title="Projection", loc="best")
ax_cor.set_xlabel('M-L axis CCF (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis CCF (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.show()

# --- Sagittal (AP vs DV) ---
fig_sag, ax_sag = plt.subplots(figsize=(8, 8))
ax_sag.scatter(mesh_vertices_mm[:, 0], mesh_vertices_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_sag.scatter(
            mm_x[mask], mm_z[mask], 
            c=color, s=10, alpha=0.9, 
            label=projection
        )

ax_sag.legend(title="Projection", loc="best")
ax_sag.set_xlabel('A-P axis CCF (mm)', fontsize=14)
ax_sag.set_ylabel('D-V axis CCF (mm)', fontsize=14)
ax_sag.invert_yaxis()
#ax_sag.invert_xaxis()
ax_sag.set_aspect('equal')
ax_sag.set_frame_on(False)
plt.show()

In [ ]:
# --- Set bregma in microns for 25um CCF ---
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

# --- Mesh: subtract bregma ---
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP
mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])  # Mirror ML to left if desired

# --- MAPseq_cells: coordinates are already in microns, subtract bregma ---
exa_pts_um = MAPseq_cells[["RC", "DV", "ML"]].values  # Already in microns
exa_pts_bregma_um = exa_pts_um - bregma_um
exa_pts_bregma_um[:, 0] = -exa_pts_bregma_um[:, 0]  # Flip AP

# --- Assign colors using proj_color ---
proj_colors = MAPseq_cells["proj_color"].fillna("#cccccc").tolist()

# --- Extract coordinates for plotting (in mm) ---
mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
exa_pts_bregma_mm = exa_pts_bregma_um / 1000.0
mm_x, mm_y, mm_z = exa_pts_bregma_mm[:, 0], exa_pts_bregma_mm[:, 2], exa_pts_bregma_mm[:, 1]  # AP, ML, DV

# --- Create hemisphere masks based on ML coordinates ---
left_hemisphere_mask = mm_y <= 0  # Left hemisphere: ML <= 0
right_hemisphere_mask = mm_y > 0   # Right hemisphere: ML > 0

# --- Coronal (ML vs DV) ---
fig_cor, ax_cor = plt.subplots(figsize=(12, 8))  # Wider to accommodate both hemispheres
ax_cor.scatter(mesh_vertices_bregma_mm[:, 2], mesh_vertices_bregma_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)
# Plot mirrored mesh (flip ML)
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1
ax_cor.scatter(mesh_vertices_bregma_mm_mirror[:, 2], mesh_vertices_bregma_mm_mirror[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_cor.scatter([], [], c=color, s=10, alpha=0.8, label=projection)

ax_cor.legend(title="Projection", loc="best")
ax_cor.scatter(mm_y, mm_z, c=proj_colors, s=10, alpha=0.9)

# Add hemisphere labels positioned relative to the actual data range
y_range = ax_cor.get_ylim()
x_range = ax_cor.get_xlim()
label_y = y_range[0] + 0.99 * (y_range[1] - y_range[0])  # Near bottom of plot
left_x = x_range[0] + 0.15 * (x_range[1] - x_range[0])   # 15% from left edge
right_x = x_range[1] - 0.15 * (x_range[1] - x_range[0])  # 15% from right edge

ax_cor.text(left_x, label_y, 'Left Hemisphere', fontsize=16, ha='center', va='center')
ax_cor.text(right_x, label_y, 'Right Hemisphere', fontsize=16, ha='center', va='center')

ax_cor.set_xlabel('M-L axis bregma (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.savefig('/results/plots/MAPseq_coronal_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/MAPseq_coronal_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Calculate common axis limits for sagittal plots ---
# Combine mesh and all cell coordinates to get global limits
all_ap_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 0],  # mesh AP
    mm_x                            # all cells AP
])
all_dv_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 1],  # mesh DV
    mm_z                            # all cells DV
])

ap_min, ap_max = all_ap_coords.min(), all_ap_coords.max()
dv_min, dv_max = all_dv_coords.min(), all_dv_coords.max()

# Add margins
ap_margin = 0.05 * (ap_max - ap_min)
dv_margin = 0.05 * (dv_max - dv_min)
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# --- Sagittal Left Hemisphere (AP vs DV) ---
fig_sag_left, ax_sag_left = plt.subplots(figsize=(8, 8))
ax_sag_left.scatter(mesh_vertices_bregma_mm[:, 0], mesh_vertices_bregma_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_sag_left.scatter([], [], c=color, s=10, alpha=0.8, label=projection)

ax_sag_left.legend(title="Projection", loc="best")
# Plot only left hemisphere cells (ML <= 0)
left_proj_colors = [proj_colors[i] for i in range(len(proj_colors)) if left_hemisphere_mask[i]]
ax_sag_left.scatter(mm_x[left_hemisphere_mask], mm_z[left_hemisphere_mask], c=left_proj_colors, s=10, alpha=0.9)
ax_sag_left.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_left.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_left.set_title('Left Hemisphere', fontsize=16, y=-0.13)
ax_sag_left.set_xlim(ap_limits)
ax_sag_left.set_ylim(dv_limits)
ax_sag_left.invert_yaxis()
ax_sag_left.invert_xaxis()
ax_sag_left.set_aspect('equal')
ax_sag_left.set_frame_on(False)
plt.savefig('/results/plots/MAPseq_sagittal_left_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/MAPseq_sagittal_left_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Sagittal Right Hemisphere (AP vs DV) ---
fig_sag_right, ax_sag_right = plt.subplots(figsize=(8, 8))
# Plot right hemisphere mesh (mirrored ML coordinates)
mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_right[:, 2] *= -1  # Mirror ML for right hemisphere
ax_sag_right.scatter(mesh_vertices_bregma_mm_right[:, 0], mesh_vertices_bregma_mm_right[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_sag_right.scatter([], [], c=color, s=10, alpha=0.8, label=projection)

ax_sag_right.legend(title="Projection", loc="best")
# Plot only right hemisphere cells (ML > 0)
right_proj_colors = [proj_colors[i] for i in range(len(proj_colors)) if right_hemisphere_mask[i]]
ax_sag_right.scatter(mm_x[right_hemisphere_mask], mm_z[right_hemisphere_mask], c=right_proj_colors, s=10, alpha=0.9)
ax_sag_right.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_right.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_right.set_title('Right Hemisphere', fontsize=16, y=-0.13)
ax_sag_right.set_xlim(ap_limits)
ax_sag_right.set_ylim(dv_limits)
ax_sag_right.invert_yaxis()
ax_sag_right.invert_xaxis()
ax_sag_right.set_aspect('equal')
ax_sag_right.set_frame_on(False)
plt.savefig('/results/plots/MAPseq_sagittal_right_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/MAPseq_sagittal_right_plot.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Set bregma in microns for 25um CCF ---
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

# --- Mesh: subtract bregma ---
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP
mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])  # Mirror ML to left if desired

# --- MAPseq_cells: coordinates are already in microns, subtract bregma ---
exa_pts_um = MAPseq_cells[["RC", "DV", "ML"]].values  # Already in microns
exa_pts_bregma_um = exa_pts_um - bregma_um
exa_pts_bregma_um[:, 0] = -exa_pts_bregma_um[:, 0]  # Flip AP

# --- Assign colors using proj_color ---
proj_colors = MAPseq_cells["proj_color"].fillna("#cccccc").tolist()

# --- Extract coordinates for plotting (in mm) ---
mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
exa_pts_bregma_mm = exa_pts_bregma_um / 1000.0
mm_x, mm_y, mm_z = exa_pts_bregma_mm[:, 0], exa_pts_bregma_mm[:, 2], exa_pts_bregma_mm[:, 1]  # AP, ML, DV

# --- Create hemisphere masks based on ML coordinates ---
left_hemisphere_mask = mm_y <= 0  # Left hemisphere: ML <= 0
right_hemisphere_mask = mm_y > 0   # Right hemisphere: ML > 0

# Helper function for bitmap mesh
def create_mesh_bitmap(mesh_x, mesh_y, x_lim, y_lim, resolution=500, alpha=mesh_alpha):
    from scipy.ndimage import gaussian_filter
    hist, x_edges, y_edges = np.histogram2d(
        mesh_x, mesh_y,
        bins=resolution,
        range=[x_lim, y_lim]
    )
    # Gaussian smoothing for antialiased, soft mesh edges
    binary_mask = (hist > 0).astype(float)
    alpha_channel = gaussian_filter(binary_mask, sigma=0.8)
    alpha_channel = np.clip(alpha_channel, 0, 1)
    bitmap = np.zeros((resolution, resolution, 4))
    bitmap[:, :, :3] = 0.35  # Gray RGB (darker)
    bitmap[:, :, 3] = alpha_channel.T * alpha
    return bitmap, [x_lim[0], x_lim[1], y_lim[0], y_lim[1]]

# --- Coronal (ML vs DV) ---
# Set limits for coronal (ML vs DV)
ml_min, ml_max = mm_y.min() - 0.1, mm_y.max() + 0.1
dv_min, dv_max = mm_z.min() - 0.1, mm_z.max() + 0.1
ml_limits = (ml_min, ml_max)
dv_limits = (dv_min, dv_max)

# Create bitmaps for coronal
bitmap_mesh_orig, extent_orig = create_mesh_bitmap(
    mesh_vertices_bregma_mm[:, 2], mesh_vertices_bregma_mm[:, 1], ml_limits, dv_limits
)
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1
bitmap_mesh_mirr, extent_mirr = create_mesh_bitmap(
    mesh_vertices_bregma_mm_mirror[:, 2], mesh_vertices_bregma_mm_mirror[:, 1], ml_limits, dv_limits
)

fig_cor, ax_cor = plt.subplots(figsize=(12, 8))  # Wider to accommodate both hemispheres
# Replace scatter with imshow for bitmap meshes
ax_cor.imshow(bitmap_mesh_orig, extent=extent_orig, origin='lower', aspect='auto', interpolation='nearest')
ax_cor.imshow(bitmap_mesh_mirr, extent=extent_mirr, origin='lower', aspect='auto', interpolation='nearest')

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_cor.scatter([], [], c=color, s=10, alpha=0.8, label=projection)

ax_cor.legend(title="Projection", loc="best")
ax_cor.scatter(mm_y, mm_z, c=proj_colors, s=10, alpha=0.9)

# Add hemisphere labels positioned relative to the actual data range
y_range = ax_cor.get_ylim()
x_range = ax_cor.get_xlim()
label_y = y_range[0] + 0.99 * (y_range[1] - y_range[0])  # Near bottom of plot
left_x = x_range[0] + 0.15 * (x_range[1] - x_range[0])   # 15% from left edge
right_x = x_range[1] - 0.15 * (x_range[1] - x_range[0])  # 15% from right edge

ax_cor.text(left_x, label_y, 'Left Hemisphere', fontsize=16, ha='center', va='center')
ax_cor.text(right_x, label_y, 'Right Hemisphere', fontsize=16, ha='center', va='center')

ax_cor.set_xlabel('M-L axis bregma (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.savefig('/results/plots/MAPseq_coronal_plot_bitmap_mesh.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/MAPseq_coronal_plot_bitmap_mesh.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Calculate common axis limits for sagittal plots ---
# Combine mesh and all cell coordinates to get global limits
all_ap_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 0],  # mesh AP
    mm_x                            # all cells AP
])
all_dv_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 1],  # mesh DV
    mm_z                            # all cells DV
])

ap_min, ap_max = all_ap_coords.min(), all_ap_coords.max()
dv_min, dv_max = all_dv_coords.min(), all_dv_coords.max()

# Add margins
ap_margin = 0.05 * (ap_max - ap_min)
dv_margin = 0.05 * (dv_max - dv_min)
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# --- Sagittal Left Hemisphere (AP vs DV) ---
# Create bitmap for left (original mesh)
bitmap_mesh_left, extent_left = create_mesh_bitmap(
    mesh_vertices_bregma_mm[:, 0], mesh_vertices_bregma_mm[:, 1], ap_limits, dv_limits
)

fig_sag_left, ax_sag_left = plt.subplots(figsize=(8, 8))
# Replace scatter with imshow
ax_sag_left.imshow(bitmap_mesh_left, extent=extent_left, origin='lower', aspect='auto', interpolation='nearest')

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_sag_left.scatter([], [], c=color, s=10, alpha=0.8, label=projection)

ax_sag_left.legend(title="Projection", loc="best")
# Plot only left hemisphere cells (ML <= 0)
left_proj_colors = [proj_colors[i] for i in range(len(proj_colors)) if left_hemisphere_mask[i]]
ax_sag_left.scatter(mm_x[left_hemisphere_mask], mm_z[left_hemisphere_mask], c=left_proj_colors, s=10, alpha=0.9)
ax_sag_left.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_left.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_left.set_title('Left Hemisphere', fontsize=16, y=-0.13)
ax_sag_left.set_xlim(ap_limits)
ax_sag_left.set_ylim(dv_limits)
ax_sag_left.invert_yaxis()
ax_sag_left.invert_xaxis()
ax_sag_left.set_aspect('equal')
ax_sag_left.set_frame_on(False)
plt.savefig('/results/plots/MAPseq_sagittal_left_plot_bitmap_mesh.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/MAPseq_sagittal_left_plot_bitmap_mesh.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Sagittal Right Hemisphere (AP vs DV) ---
# Create bitmap for right (mirrored mesh)
mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_right[:, 2] *= -1  # Mirror ML for right hemisphere
bitmap_mesh_right, extent_right = create_mesh_bitmap(
    mesh_vertices_bregma_mm_right[:, 0], mesh_vertices_bregma_mm_right[:, 1], ap_limits, dv_limits
)

fig_sag_right, ax_sag_right = plt.subplots(figsize=(8, 8))
# Replace scatter with imshow
ax_sag_right.imshow(bitmap_mesh_right, extent=extent_right, origin='lower', aspect='auto', interpolation='nearest')

# Plot each projection type separately with mapped colors
for projection, color in topProj_colors_hex.items():
    mask = MAPseq_cells["top_projection"] == projection
    if mask.any():
        ax_sag_right.scatter([], [], c=color, s=10, alpha=0.8, label=projection)

ax_sag_right.legend(title="Projection", loc="best")
# Plot only right hemisphere cells (ML > 0)
right_proj_colors = [proj_colors[i] for i in range(len(proj_colors)) if right_hemisphere_mask[i]]
ax_sag_right.scatter(mm_x[right_hemisphere_mask], mm_z[right_hemisphere_mask], c=right_proj_colors, s=10, alpha=0.9)
ax_sag_right.set_xlabel('A-P axis bregma (mm)', fontsize=14)
ax_sag_right.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_sag_right.set_title('Right Hemisphere', fontsize=16, y=-0.13)
ax_sag_right.set_xlim(ap_limits)
ax_sag_right.set_ylim(dv_limits)
ax_sag_right.invert_yaxis()
ax_sag_right.invert_xaxis()
ax_sag_right.set_aspect('equal')
ax_sag_right.set_frame_on(False)
plt.savefig('/results/plots/MAPseq_sagittal_right_plot_bitmap_mesh.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/MAPseq_sagittal_right_plot_bitmap_mesh.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# read in Dbh_Th_Slc18a2_logcounts_adj_Dbh.csv file from results directory
logcounts_path = "/data/PK_MAPseqBARseq_LC-NEpaper_files/Dbh_Th_Slc18a2_logcounts_adj_Dbh.csv"
logcounts_df = pd.read_csv(logcounts_path, index_col=0)
# Display first few rows of logcounts_df
display(logcounts_df.head())

# Display first few rows from MAPseq_cells to check for 'uid' column
display(MAPseq_cells.head())

In [ ]:
# 1) Extract uid from logcounts_df index and merge with MAPseq_cells

# The logcounts_df index looks like "brain3|5_17_380460" — uid is after the "|"
logcounts_df = logcounts_df.copy()
logcounts_df["uid"] = logcounts_df.index.str.split("|", n=1).str[1]

# MAPseq_cells uid is in the "uid" column already
# Check types match
print(f"logcounts_df uid examples: {logcounts_df['uid'].head(3).tolist()}")
print(f"MAPseq_cells uid examples: {MAPseq_cells['uid'].head(3).tolist()}")

# Convert both to string to be safe
logcounts_df["uid"] = logcounts_df["uid"].astype(str)
MAPseq_cells["uid"] = MAPseq_cells["uid"].astype(str)

# 2) Sanity checks before merge
uids_logcounts = set(logcounts_df["uid"])
uids_mapseq = set(MAPseq_cells["uid"])

shared = uids_logcounts & uids_mapseq
only_logcounts = uids_logcounts - uids_mapseq
only_mapseq = uids_mapseq - uids_logcounts

print(f"\n--- Merge diagnostics ---")
print(f"Cells in logcounts_df:  {len(uids_logcounts)}")
print(f"Cells in MAPseq_cells:  {len(uids_mapseq)}")
print(f"Shared (will be plotted): {len(shared)}")
print(f"In logcounts only (no coordinates, cannot plot): {len(only_logcounts)}")
print(f"In MAPseq only (no expression data): {len(only_mapseq)}")

if only_logcounts:
    print(f"\n  Logcounts-only uids (first 10): {list(only_logcounts)[:10]}")
    # Check batch distribution of unmatched cells
    unmatched = logcounts_df[logcounts_df["uid"].isin(only_logcounts)]
    print(f"  Batch distribution of unmatched: {unmatched['batch'].value_counts().to_dict()}")

if only_mapseq:
    print(f"\n  MAPseq-only uids (first 10): {list(only_mapseq)[:10]}")

# 3) Merge: inner join on uid
merged_df = logcounts_df.merge(
    MAPseq_cells[["uid", "RC", "DV", "ML", "top_projection", "proj_color"]],
    on="uid",
    how="inner"
)
print(f"\nMerged dataframe: {merged_df.shape[0]} cells x {merged_df.shape[1]} columns")
print(merged_df.head())

In [ ]:
# ============================================================
# 4) Convert coordinates to bregma-relative mm
#    (same transform as your existing plotting code)
# ============================================================
import matplotlib.colors as mcolors
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

coords_um = merged_df[["RC", "DV", "ML"]].values
coords_bregma_um = coords_um - bregma_um
coords_bregma_um[:, 0] = -coords_bregma_um[:, 0]  # Flip AP

coords_bregma_mm = coords_bregma_um / 1000.0

merged_df["AP_mm"] = coords_bregma_mm[:, 0]
merged_df["ML_mm"] = coords_bregma_mm[:, 2]  # ML is 3rd coord
merged_df["DV_mm"] = coords_bregma_mm[:, 1]

# ============================================================
# 5) Plot each gene on coronal + sagittal views with viridis
#    Same mesh bitmap approach as your existing code
# ============================================================
genes_to_plot = ["Dbh", "Th", "Slc18a2"]

# Hemisphere masks
left_mask = merged_df["ML_mm"] <= 0
right_mask = merged_df["ML_mm"] > 0

for gene in genes_to_plot:
    expr = merged_df[gene].values
    
    # Use consistent vmin/vmax across views for each gene
    # vmin=0 since logcounts are non-negative
    vmin = 0
    vmax = np.percentile(expr[expr > 0], 95) if (expr > 0).any() else 1
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    
    print(f"\n--- {gene} ---")
    print(f"  Expressing cells: {(expr > 0).sum()} / {len(expr)} ({(expr > 0).mean():.1%})")
    print(f"  Range: [{expr.min():.3f}, {expr.max():.3f}], vmax (95th pctile of expressing): {vmax:.3f}")
    
    # Sort so high-expression cells plot on top (zero-expression in background)
    sort_idx = np.argsort(expr)
    
    # --- Coronal view (ML vs DV) ---
    fig_cor, ax_cor = plt.subplots(figsize=(12, 8))
    
    # Mesh bitmaps (reuse your existing bitmap variables)
    ax_cor.imshow(bitmap_mesh_orig, extent=extent_orig, origin='lower', aspect='equal', interpolation='nearest')
    ax_cor.imshow(bitmap_mesh_mirr, extent=extent_mirr, origin='lower', aspect='equal', interpolation='nearest')
    
    sc = ax_cor.scatter(
        merged_df["ML_mm"].values[sort_idx],
        merged_df["DV_mm"].values[sort_idx],
        c=expr[sort_idx],
        cmap="viridis",
        norm=norm,
        s=12,
        alpha=0.9
    )
    cbar = plt.colorbar(sc, ax=ax_cor, shrink=0.7, pad=0.02)
    cbar.set_label(f"{gene} log2(CPM+1)", fontsize=12)
    
    ax_cor.set_xlabel("M-L axis bregma (mm)", fontsize=14)
    ax_cor.set_ylabel("D-V axis bregma (mm)", fontsize=14)
    ax_cor.set_title(f"{gene} expression — Coronal", fontsize=16)
    ax_cor.invert_yaxis()
    ax_cor.set_aspect("equal")
    ax_cor.set_frame_on(False)
    
    plt.savefig(f"/results/plots/{gene}_expression_coronal_bitmap_mesh.pdf", format="pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"/results/plots/{gene}_expression_coronal_bitmap_mesh.svg", format="svg", bbox_inches="tight")
    plt.show()
    
    # --- Sagittal Left (AP vs DV) ---
    left_idx = np.where(left_mask)[0]
    left_expr = expr[left_idx]
    left_sort = left_idx[np.argsort(left_expr)]
    
    fig_sag_l, ax_sag_l = plt.subplots(figsize=(8, 8))
    ax_sag_l.imshow(bitmap_mesh_left, extent=extent_left, origin='lower', aspect='equal', interpolation='nearest')
    
    sc_l = ax_sag_l.scatter(
        merged_df["AP_mm"].values[left_sort],
        merged_df["DV_mm"].values[left_sort],
        c=expr[left_sort],
        cmap="viridis",
        norm=norm,
        s=12,
        alpha=0.9
    )
    cbar_l = plt.colorbar(sc_l, ax=ax_sag_l, shrink=0.7, pad=0.02)
    cbar_l.set_label(f"{gene} log2(CPM+1)", fontsize=12)
    
    ax_sag_l.set_xlabel("A-P bregma axis (mm)", fontsize=14)
    ax_sag_l.set_ylabel("D-V bregma axis (mm)", fontsize=14)
    ax_sag_l.set_title(f"{gene} — Left Hemisphere", fontsize=16, y=-0.13)
    ax_sag_l.set_xlim(ap_limits)
    ax_sag_l.set_ylim(dv_limits)
    ax_sag_l.invert_yaxis()
    ax_sag_l.invert_xaxis()
    ax_sag_l.set_aspect("equal")
    ax_sag_l.set_frame_on(False)
    
    plt.savefig(f"/results/plots/{gene}_expression_sagittal_left_bitmap_mesh.pdf", format="pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"/results/plots/{gene}_expression_sagittal_left_bitmap_mesh.svg", format="svg", bbox_inches="tight")
    plt.show()
    
    # --- Sagittal Right (AP vs DV) ---
    right_idx = np.where(right_mask)[0]
    right_expr = expr[right_idx]
    right_sort = right_idx[np.argsort(right_expr)]
    
    fig_sag_r, ax_sag_r = plt.subplots(figsize=(8, 8))
    ax_sag_r.imshow(bitmap_mesh_right, extent=extent_right, origin='lower', aspect='equal', interpolation='nearest')
    
    sc_r = ax_sag_r.scatter(
        merged_df["AP_mm"].values[right_sort],
        merged_df["DV_mm"].values[right_sort],
        c=expr[right_sort],
        cmap="viridis",
        norm=norm,
        s=12,
        alpha=0.9
    )
    cbar_r = plt.colorbar(sc_r, ax=ax_sag_r, shrink=0.7, pad=0.02)
    cbar_r.set_label(f"{gene} log2(CPM+1)", fontsize=12)
    
    ax_sag_r.set_xlabel("A-P axis bregma (mm)", fontsize=14)
    ax_sag_r.set_ylabel("D-V bregma axis (mm)", fontsize=14)
    ax_sag_r.set_title(f"{gene} — Right Hemisphere", fontsize=16, y=-0.13)
    ax_sag_r.set_xlim(ap_limits)
    ax_sag_r.set_ylim(dv_limits)
    ax_sag_r.invert_yaxis()
    ax_sag_r.invert_xaxis()
    ax_sag_r.set_aspect("equal")
    ax_sag_r.set_frame_on(False)
    
    plt.savefig(f"/results/plots/{gene}_expression_sagittal_right_bitmap_mesh.pdf", format="pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"/results/plots/{gene}_expression_sagittal_right_bitmap_mesh.svg", format="svg", bbox_inches="tight")
    plt.show()

In [ ]:
bregma_um = np.array([216, 18, 228]) * 25  # [RC, DV, ML] in um

# ---- Cells (merged_df) -> bregma mm ----
coords_um = merged_df[["RC", "DV", "ML"]].values
coords_bregma_um = coords_um - bregma_um
coords_bregma_um[:, 0] = -coords_bregma_um[:, 0]  # Flip AP (RC)

coords_bregma_mm = coords_bregma_um / 1000.0
merged_df["AP_mm"] = coords_bregma_mm[:, 0]
merged_df["DV_mm"] = coords_bregma_mm[:, 1]
merged_df["ML_mm"] = coords_bregma_mm[:, 2]

# ---- Mesh vertices -> bregma mm ----
# Requires: mesh_vertices (um) shaped (N,3) in [RC, DV, ML]
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP

# Match your "full rendering" behavior: force mesh to left hemisphere then mirror when plotting
mirror_mesh_to_left = True
if mirror_mesh_to_left:
    mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])

mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1  # mirrored for right hemisphere

# Mesh styling (use existing vars if present; otherwise defaults)
mesh_alpha_use = float(globals().get("mesh_alpha", 0.08))
mesh_dot_size_use = float(globals().get("dot_size", 0.2))  # keep very small if mesh is dense

# Optional: if files get huge, set mesh_step=5 or 10 (keeps vector, fewer points)
mesh_step = int(globals().get("mesh_step", 1))

# ============================================================
# 5) Plot each gene on coronal + sagittal views with viridis
#    Vector "full rendering" mesh background (scatter), no bitmap imshow
# ============================================================
genes_to_plot = ["Dbh", "Th", "Slc18a2"]

left_mask = merged_df["ML_mm"] <= 0
right_mask = merged_df["ML_mm"] > 0

# ---- Axis limits (computed once; consistent across genes) ----
# Coronal limits (ML vs DV): include BOTH mirrored mesh and all cells
all_ml = np.concatenate([
    mesh_vertices_bregma_mm[::mesh_step, 2],
    mesh_vertices_bregma_mm_mirror[::mesh_step, 2],
    merged_df["ML_mm"].values
])
all_dv = np.concatenate([
    mesh_vertices_bregma_mm[::mesh_step, 1],
    mesh_vertices_bregma_mm_mirror[::mesh_step, 1],
    merged_df["DV_mm"].values
])

ml_min, ml_max = all_ml.min(), all_ml.max()
dv_min, dv_max = all_dv.min(), all_dv.max()
ml_margin = 0.05 * (ml_max - ml_min) if ml_max > ml_min else 0.1
dv_margin = 0.05 * (dv_max - dv_min) if dv_max > dv_min else 0.1
ml_limits = (ml_min - ml_margin, ml_max + ml_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# Sagittal limits (AP vs DV): include mesh + all cells
all_ap = np.concatenate([mesh_vertices_bregma_mm[::mesh_step, 0], merged_df["AP_mm"].values])
all_dv2 = np.concatenate([mesh_vertices_bregma_mm[::mesh_step, 1], merged_df["DV_mm"].values])

ap_min, ap_max = all_ap.min(), all_ap.max()
dv2_min, dv2_max = all_dv2.min(), all_dv2.max()
ap_margin = 0.05 * (ap_max - ap_min) if ap_max > ap_min else 0.1
dv2_margin = 0.05 * (dv2_max - dv2_min) if dv2_max > dv2_min else 0.1
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits_sag = (dv2_min - dv2_margin, dv2_max + dv2_margin)

for gene in genes_to_plot:
    expr = merged_df[gene].values

    vmin = 0
    vmax = np.percentile(expr[expr > 0], 95) if (expr > 0).any() else 1
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    print(f"\n--- {gene} ---")
    print(f"  Expressing cells: {(expr > 0).sum()} / {len(expr)} ({(expr > 0).mean():.1%})")
    print(f"  Range: [{expr.min():.3f}, {expr.max():.3f}], vmax (95th pctile of expressing): {vmax:.3f}")

    sort_idx = np.argsort(expr)

    # --- Coronal (ML vs DV) ---
    fig_cor, ax_cor = plt.subplots(figsize=(12, 8))

    # Mesh background as dense vector points (left + mirrored right)
    ax_cor.scatter(
        mesh_vertices_bregma_mm[::mesh_step, 2],
        mesh_vertices_bregma_mm[::mesh_step, 1],
        color="gray", s=mesh_dot_size_use, alpha=mesh_alpha_use, linewidths=0
    )
    ax_cor.scatter(
        mesh_vertices_bregma_mm_mirror[::mesh_step, 2],
        mesh_vertices_bregma_mm_mirror[::mesh_step, 1],
        color="gray", s=mesh_dot_size_use, alpha=mesh_alpha_use, linewidths=0
    )

    sc = ax_cor.scatter(
        merged_df["ML_mm"].values[sort_idx],
        merged_df["DV_mm"].values[sort_idx],
        c=expr[sort_idx],
        cmap="viridis",
        norm=norm,
        s=12,
        alpha=0.9
    )
    cbar = plt.colorbar(sc, ax=ax_cor, shrink=0.7, pad=0.02)
    cbar.set_label(f"{gene} log2(CPM+1)", fontsize=12)

    ax_cor.set_xlabel("M-L axis bregma (mm)", fontsize=14)
    ax_cor.set_ylabel("D-V axis bregma (mm)", fontsize=14)
    ax_cor.set_title(f"{gene} expression — Coronal", fontsize=16)

    ax_cor.set_xlim(ml_limits)
    ax_cor.set_ylim(dv_limits)
    ax_cor.invert_yaxis()
    ax_cor.set_aspect("equal", adjustable="box")
    ax_cor.autoscale(False)
    ax_cor.set_frame_on(False)

    plt.savefig(f"/results/plots/{gene}_expression_coronal_fullmesh.pdf",
                format="pdf", dpi=300, bbox_inches="tight", transparent=True)
    plt.savefig(f"/results/plots/{gene}_expression_coronal_fullmesh.svg",
                format="svg", bbox_inches="tight", transparent=True)
    plt.show()

    # --- Sagittal Left (AP vs DV) ---
    left_idx = np.where(left_mask)[0]
    left_sort = left_idx[np.argsort(expr[left_idx])]

    fig_sag_l, ax_sag_l = plt.subplots(figsize=(8, 8))

    # Mesh (left)
    ax_sag_l.scatter(
        mesh_vertices_bregma_mm[::mesh_step, 0],
        mesh_vertices_bregma_mm[::mesh_step, 1],
        color="gray", s=mesh_dot_size_use, alpha=mesh_alpha_use, linewidths=0
    )

    sc_l = ax_sag_l.scatter(
        merged_df["AP_mm"].values[left_sort],
        merged_df["DV_mm"].values[left_sort],
        c=expr[left_sort],
        cmap="viridis",
        norm=norm,
        s=12,
        alpha=0.9
    )
    cbar_l = plt.colorbar(sc_l, ax=ax_sag_l, shrink=0.7, pad=0.02)
    cbar_l.set_label(f"{gene} log2(CPM+1)", fontsize=12)

    ax_sag_l.set_xlabel("A-P bregma axis (mm)", fontsize=14)
    ax_sag_l.set_ylabel("D-V bregma axis (mm)", fontsize=14)
    ax_sag_l.set_title(f"{gene} — Left Hemisphere", fontsize=16, y=-0.13)

    ax_sag_l.set_xlim(ap_limits)
    ax_sag_l.set_ylim(dv_limits_sag)
    ax_sag_l.invert_yaxis()
    ax_sag_l.invert_xaxis()
    ax_sag_l.set_aspect("equal", adjustable="box")
    ax_sag_l.autoscale(False)
    ax_sag_l.set_frame_on(False)

    plt.savefig(f"/results/plots/{gene}_expression_sagittal_left_fullmesh.pdf",
                format="pdf", dpi=300, bbox_inches="tight", transparent=True)
    plt.savefig(f"/results/plots/{gene}_expression_sagittal_left_fullmesh.svg",
                format="svg", bbox_inches="tight", transparent=True)
    plt.show()

    # --- Sagittal Right (AP vs DV) ---
    right_idx = np.where(right_mask)[0]
    right_sort = right_idx[np.argsort(expr[right_idx])]

    fig_sag_r, ax_sag_r = plt.subplots(figsize=(8, 8))

    # Mesh (right) = mirror ML for “right hemisphere view” like your example
    mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
    mesh_vertices_bregma_mm_right[:, 2] *= -1

    ax_sag_r.scatter(
        mesh_vertices_bregma_mm_right[::mesh_step, 0],
        mesh_vertices_bregma_mm_right[::mesh_step, 1],
        color="gray", s=mesh_dot_size_use, alpha=mesh_alpha_use, linewidths=0
    )

    sc_r = ax_sag_r.scatter(
        merged_df["AP_mm"].values[right_sort],
        merged_df["DV_mm"].values[right_sort],
        c=expr[right_sort],
        cmap="viridis",
        norm=norm,
        s=12,
        alpha=0.9
    )
    cbar_r = plt.colorbar(sc_r, ax=ax_sag_r, shrink=0.7, pad=0.02)
    cbar_r.set_label(f"{gene} log2(CPM+1)", fontsize=12)

    ax_sag_r.set_xlabel("A-P axis bregma (mm)", fontsize=14)
    ax_sag_r.set_ylabel("D-V bregma axis (mm)", fontsize=14)
    ax_sag_r.set_title(f"{gene} — Right Hemisphere", fontsize=16, y=-0.13)

    ax_sag_r.set_xlim(ap_limits)
    ax_sag_r.set_ylim(dv_limits_sag)
    ax_sag_r.invert_yaxis()
    ax_sag_r.invert_xaxis()
    ax_sag_r.set_aspect("equal", adjustable="box")
    ax_sag_r.autoscale(False)
    ax_sag_r.set_frame_on(False)

    plt.savefig(f"/results/plots/{gene}_expression_sagittal_right_fullmesh.pdf",
                format="pdf", dpi=300, bbox_inches="tight", transparent=True)
    plt.savefig(f"/results/plots/{gene}_expression_sagittal_right_fullmesh.svg",
                format="svg", bbox_inches="tight", transparent=True)
    plt.show()

In [ ]:
# --- Set bregma in microns for 25um CCF ---
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

# --- Mesh: subtract bregma ---
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP
mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])  # Mirror ML to left if desired

# --- BARseq_cells: coordinates are already in microns, subtract bregma ---
exa_pts_um = BARseq_plotting[["RC", "DV", "ML"]].values  # Already in microns
exa_pts_bregma_um = exa_pts_um - bregma_um
exa_pts_bregma_um[:, 0] = -exa_pts_bregma_um[:, 0]  # Flip AP

# --- Assign colors using proj_color ---
proj_colors = BARseq_plotting["proj_color"].fillna("#cccccc").tolist()

# --- Extract coordinates for plotting (in mm) ---
mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
exa_pts_bregma_mm = exa_pts_bregma_um / 1000.0
mm_x, mm_y, mm_z = exa_pts_bregma_mm[:, 0], exa_pts_bregma_mm[:, 2], exa_pts_bregma_mm[:, 1]  # AP, ML, DV

# --- Create hemisphere masks based on ML coordinates ---
left_hemisphere_mask = mm_y <= 0  # Left hemisphere: ML <= 0
right_hemisphere_mask = mm_y > 0   # Right hemisphere: ML > 0

# --- Create masks for cell types ---
barcoded_mask = BARseq_plotting["proj_color"] == "#0000FF"  # Blue cells (barcoded)
ne_mask = BARseq_plotting["proj_color"] == "#FF0000"  # Red cells (all NE)

# --- Coronal (ML vs DV) ---
fig_cor, ax_cor = plt.subplots(figsize=(12, 8))  # Wider to accommodate both hemispheres
# Combine original and mirrored mesh into one scatter to avoid plotting twice
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1
mesh_combined = np.concatenate([mesh_vertices_bregma_mm[:, [2,1]], mesh_vertices_bregma_mm_mirror[:, [2,1]]])
ax_cor.scatter(mesh_combined[:, 0], mesh_combined[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for cell types
ax_cor.scatter([], [], c="#0000FF", s=10, alpha=0.8, label="Barcoded cells")
ax_cor.scatter([], [], c="#FF0000", s=10, alpha=0.8, label="NE cells")

ax_cor.legend(title="Cell Type", loc="best")
# Plot NE cells first (red), then barcoded cells (blue) on top
ax_cor.scatter(mm_y[ne_mask], mm_z[ne_mask], c="#FF0000", s=2, alpha=0.5)
ax_cor.scatter(mm_y[barcoded_mask], mm_z[barcoded_mask], c="#0000FF", s=2, alpha=0.5)

# Add hemisphere labels positioned relative to the actual data range
y_range = ax_cor.get_ylim()
x_range = ax_cor.get_xlim()
label_y = y_range[0] + 0.99 * (y_range[1] - y_range[0])  # Near bottom of plot
left_x = x_range[0] + 0.15 * (x_range[1] - x_range[0])   # 15% from left edge
right_x = x_range[1] - 0.15 * (x_range[1] - x_range[0])  # 15% from right edge

ax_cor.text(left_x, label_y, 'Left Hemisphere', fontsize=16, ha='center', va='center')
ax_cor.text(right_x, label_y, 'Right Hemisphere', fontsize=16, ha='center', va='center')

ax_cor.set_xlabel('M-L axis bregma (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.savefig('/results/plots/BARseq_coronal_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_coronal_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Calculate common axis limits for sagittal plots ---
# Combine mesh and all cell coordinates to get global limits
all_ap_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 0],  # mesh AP
    mm_x                            # all cells AP
])
all_dv_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 1],  # mesh DV
    mm_z                            # all cells DV
])

ap_min, ap_max = all_ap_coords.min(), all_ap_coords.max()
dv_min, dv_max = all_dv_coords.min(), all_dv_coords.max()

# Add margins
ap_margin = 0.05 * (ap_max - ap_min)
dv_margin = 0.05 * (dv_max - dv_min)
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# --- Sagittal Left Hemisphere (AP vs DV) ---
fig_sag_left, ax_sag_left = plt.subplots(figsize=(8, 8))
ax_sag_left.scatter(mesh_vertices_bregma_mm[:, 0], mesh_vertices_bregma_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for cell types
ax_sag_left.scatter([], [], c="#0000FF", s=10, alpha=0.8, label="Barcoded cells")
ax_sag_left.scatter([], [], c="#FF0000", s=10, alpha=0.8, label="NE cells")

ax_sag_left.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only left hemisphere cells: NE first, then barcoded on top
left_ne_mask = left_hemisphere_mask & ne_mask
left_barcoded_mask = left_hemisphere_mask & barcoded_mask
ax_sag_left.scatter(mm_x[left_ne_mask], mm_z[left_ne_mask], c="#FF0000", s=2, alpha=0.5)
ax_sag_left.scatter(mm_x[left_barcoded_mask], mm_z[left_barcoded_mask], c="#0000FF", s=2, alpha=0.5)
ax_sag_left.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_left.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_left.set_title('Left Hemisphere', fontsize=16, y=-0.13)
ax_sag_left.set_xlim(ap_limits)
ax_sag_left.set_ylim(dv_limits)
ax_sag_left.invert_yaxis()
ax_sag_left.invert_xaxis()
ax_sag_left.set_aspect('equal')
ax_sag_left.set_frame_on(False)
plt.savefig('/results/plots/BARseq_sagittal_left_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_sagittal_left_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Sagittal Right Hemisphere (AP vs DV) ---
fig_sag_right, ax_sag_right = plt.subplots(figsize=(8, 8))
# Plot right hemisphere mesh (mirrored ML coordinates)
mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_right[:, 2] *= -1  # Mirror ML for right hemisphere
ax_sag_right.scatter(mesh_vertices_bregma_mm_right[:, 0], mesh_vertices_bregma_mm_right[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for cell types
ax_sag_right.scatter([], [], c="#0000FF", s=10, alpha=0.8, label="Barcoded cells")
ax_sag_right.scatter([], [], c="#FF0000", s=10, alpha=0.8, label="NE cells")

ax_sag_right.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only right hemisphere cells: NE first, then barcoded on top
right_ne_mask = right_hemisphere_mask & ne_mask
right_barcoded_mask = right_hemisphere_mask & barcoded_mask
ax_sag_right.scatter(mm_x[right_ne_mask], mm_z[right_ne_mask], c="#FF0000", s=2, alpha=0.5)
ax_sag_right.scatter(mm_x[right_barcoded_mask], mm_z[right_barcoded_mask], c="#0000FF", s=2, alpha=0.5)
ax_sag_right.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_right.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_right.set_title('Right Hemisphere', fontsize=16, y=-0.13)
ax_sag_right.set_xlim(ap_limits)
ax_sag_right.set_ylim(dv_limits)
ax_sag_right.invert_yaxis()
ax_sag_right.invert_xaxis()
ax_sag_right.set_aspect('equal')
ax_sag_right.set_frame_on(False)
plt.savefig('/results/plots/BARseq_sagittal_right_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_sagittal_right_plot.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Set bregma in microns for 25um CCF ---
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

# --- Mesh: subtract bregma ---
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP
mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])  # Mirror ML to left if desired

# --- BARseq_cells: coordinates are already in microns, subtract bregma ---
exa_pts_um = BARseq_batch[["RC", "DV", "ML"]].values  # Use BARseq_batch
exa_pts_bregma_um = exa_pts_um - bregma_um
exa_pts_bregma_um[:, 0] = -exa_pts_bregma_um[:, 0]  # Flip AP

# --- Assign colors using proj_color ---
proj_colors = BARseq_batch["proj_color"].fillna("#cccccc").tolist()

# --- Extract coordinates for plotting (in mm) ---
mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
exa_pts_bregma_mm = exa_pts_bregma_um / 1000.0
mm_x, mm_y, mm_z = exa_pts_bregma_mm[:, 0], exa_pts_bregma_mm[:, 2], exa_pts_bregma_mm[:, 1]  # AP, ML, DV

# --- Create hemisphere masks based on ML coordinates ---
left_hemisphere_mask = mm_y <= 0  # Left hemisphere: ML <= 0
right_hemisphere_mask = mm_y > 0   # Right hemisphere: ML > 0

# --- Create masks for cell types based on boolean_color_map ---
# Note: Using boolean columns from BARseq_batch for four categories

# --- Coronal (ML vs DV) ---
fig_cor, ax_cor = plt.subplots(figsize=(12, 8))  # Wider to accommodate both hemispheres
# Combine original and mirrored mesh into one scatter to avoid plotting twice
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1
mesh_combined = np.concatenate([mesh_vertices_bregma_mm[:, [2,1]], mesh_vertices_bregma_mm_mirror[:, [2,1]]])
ax_cor.scatter(mesh_combined[:, 0], mesh_combined[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for cell types (four categories)
for label, color in boolean_color_map.items():
    ax_cor.scatter([], [], c=color, s=10, alpha=0.8, label=label.replace("_", " ").title())  # Format labels nicely

ax_cor.legend(title="Cell Type", loc="best")
# Plot cells in order: light colors first, then dark on top (similar to template's NE then barcoded)
for label, color in boolean_color_map.items():
    mask = BARseq_batch[label]
    ax_cor.scatter(mm_y[mask], mm_z[mask], c=color, s=1, alpha=0.5)

# Add hemisphere labels positioned relative to the actual data range
y_range = ax_cor.get_ylim()
x_range = ax_cor.get_xlim()
label_y = y_range[0] + 0.99 * (y_range[1] - y_range[0])  # Near bottom of plot
left_x = x_range[0] + 0.15 * (x_range[1] - x_range[0])   # 15% from left edge
right_x = x_range[1] - 0.15 * (x_range[1] - x_range[0])  # 15% from right edge

ax_cor.text(left_x, label_y, 'Left Hemisphere', fontsize=16, ha='center', va='center')
ax_cor.text(right_x, label_y, 'Right Hemisphere', fontsize=16, ha='center', va='center')

ax_cor.set_xlabel('M-L axis bregma (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.savefig('/results/plots/BARseq_batch_coronal_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_batch_coronal_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Calculate common axis limits for sagittal plots ---
# Combine mesh and all cell coordinates to get global limits
all_ap_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 0],  # mesh AP
    mm_x                            # all cells AP
])
all_dv_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 1],  # mesh DV
    mm_z                            # all cells DV
])

ap_min, ap_max = all_ap_coords.min(), all_ap_coords.max()
dv_min, dv_max = all_dv_coords.min(), all_dv_coords.max()

# Add margins
ap_margin = 0.05 * (ap_max - ap_min)
dv_margin = 0.05 * (dv_max - dv_min)
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# --- Sagittal Left Hemisphere (AP vs DV) ---
fig_sag_left, ax_sag_left = plt.subplots(figsize=(8, 8))
ax_sag_left.scatter(mesh_vertices_bregma_mm[:, 0], mesh_vertices_bregma_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for cell types (four categories)
for label, color in boolean_color_map.items():
    ax_sag_left.scatter([], [], c=color, s=10, alpha=0.8, label=label.replace("_", " ").title())  # Format labels nicely

ax_sag_left.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only left hemisphere cells: light colors first, then dark on top
for label, color in boolean_color_map.items():
    mask = BARseq_batch[label] & left_hemisphere_mask
    ax_sag_left.scatter(mm_x[mask], mm_z[mask], c=color, s=1, alpha=0.5)
ax_sag_left.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_left.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_left.set_title('Left Hemisphere', fontsize=16, y=-0.13)
ax_sag_left.set_xlim(ap_limits)
ax_sag_left.set_ylim(dv_limits)
ax_sag_left.invert_yaxis()
ax_sag_left.invert_xaxis()
ax_sag_left.set_aspect('equal')
ax_sag_left.set_frame_on(False)
plt.savefig('/results/plots/BARseq_batch_sagittal_left_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_batch_sagittal_left_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Sagittal Right Hemisphere (AP vs DV) ---
fig_sag_right, ax_sag_right = plt.subplots(figsize=(8, 8))
# Plot right hemisphere mesh (mirrored ML coordinates)
mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_right[:, 2] *= -1  # Mirror ML for right hemisphere
ax_sag_right.scatter(mesh_vertices_bregma_mm_right[:, 0], mesh_vertices_bregma_mm_right[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for cell types (four categories)
for label, color in boolean_color_map.items():
    ax_sag_right.scatter([], [], c=color, s=10, alpha=0.8, label=label.replace("_", " ").title())  # Format labels nicely

ax_sag_right.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only right hemisphere cells: light colors first, then dark on top
for label, color in boolean_color_map.items():
    mask = BARseq_batch[label] & right_hemisphere_mask
    ax_sag_right.scatter(mm_x[mask], mm_z[mask], c=color, s=1, alpha=0.5)
ax_sag_right.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_right.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_right.set_title('Right Hemisphere', fontsize=16, y=-0.13)
ax_sag_right.set_xlim(ap_limits)
ax_sag_right.set_ylim(dv_limits)
ax_sag_right.invert_yaxis()
ax_sag_right.invert_xaxis()
ax_sag_right.set_aspect('equal')
ax_sag_right.set_frame_on(False)
plt.savefig('/results/plots/BARseq_batch_sagittal_right_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_batch_sagittal_right_plot.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Set bregma in microns for 25um CCF ---
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

# --- Mesh: subtract bregma ---
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP
mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])  # Mirror ML to left if desired

# --- BARseq_cells: coordinates are already in microns, subtract bregma ---
exa_pts_um = BARseq_batch[["RC", "DV", "ML"]].values  # Use BARseq_batch
exa_pts_bregma_um = exa_pts_um - bregma_um
exa_pts_bregma_um[:, 0] = -exa_pts_bregma_um[:, 0]  # Flip AP

# --- Assign colors using proj_color ---
proj_colors = BARseq_batch["proj_color"].fillna("#cccccc").tolist()

# --- Extract coordinates for plotting (in mm) ---
mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
exa_pts_bregma_mm = exa_pts_bregma_um / 1000.0
mm_x, mm_y, mm_z = exa_pts_bregma_mm[:, 0], exa_pts_bregma_mm[:, 2], exa_pts_bregma_mm[:, 1]  # AP, ML, DV

# --- Create hemisphere masks based on ML coordinates ---
left_hemisphere_mask = mm_y <= 0  # Left hemisphere: ML <= 0
right_hemisphere_mask = mm_y > 0   # Right hemisphere: ML > 0

# --- Create masks for cell types based on boolean_color_map ---
# Note: Using boolean columns from BARseq_batch for four categories

# Helper function for bitmap mesh
def create_mesh_bitmap(mesh_x, mesh_y, x_lim, y_lim, resolution=500, alpha=mesh_alpha):
    from scipy.ndimage import gaussian_filter
    hist, x_edges, y_edges = np.histogram2d(
        mesh_x, mesh_y,
        bins=resolution,
        range=[x_lim, y_lim]
    )
    # Gaussian smoothing for antialiased, soft mesh edges
    binary_mask = (hist > 0).astype(float)
    alpha_channel = gaussian_filter(binary_mask, sigma=0.8)
    alpha_channel = np.clip(alpha_channel, 0, 1)
    bitmap = np.zeros((resolution, resolution, 4))
    bitmap[:, :, :3] = 0.35  # Gray RGB (darker)
    bitmap[:, :, 3] = alpha_channel.T * alpha
    return bitmap, [x_lim[0], x_lim[1], y_lim[0], y_lim[1]]

# --- Coronal (ML vs DV) ---
# Set limits for coronal (ML vs DV)
ml_min, ml_max = mm_y.min() - 0.1, mm_y.max() + 0.1
dv_min, dv_max = mm_z.min() - 0.1, mm_z.max() + 0.1
ml_limits = (ml_min, ml_max)
dv_limits = (dv_min, dv_max)

# Create bitmaps for coronal
bitmap_mesh_orig, extent_orig = create_mesh_bitmap(
    mesh_vertices_bregma_mm[:, 2], mesh_vertices_bregma_mm[:, 1], ml_limits, dv_limits
)
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1
bitmap_mesh_mirr, extent_mirr = create_mesh_bitmap(
    mesh_vertices_bregma_mm_mirror[:, 2], mesh_vertices_bregma_mm_mirror[:, 1], ml_limits, dv_limits
)

fig_cor, ax_cor = plt.subplots(figsize=(12, 8))  # Wider to accommodate both hemispheres
# Replace scatter with imshow for bitmap meshes
ax_cor.imshow(bitmap_mesh_orig, extent=extent_orig, origin='lower', aspect='auto', interpolation='nearest')
ax_cor.imshow(bitmap_mesh_mirr, extent=extent_mirr, origin='lower', aspect='auto', interpolation='nearest')

# Create legend entries for cell types (four categories)
for label, color in boolean_color_map.items():
    ax_cor.scatter([], [], c=color, s=10, alpha=0.8, label=label.replace("_", " ").title())  # Format labels nicely

ax_cor.legend(title="Cell Type", loc="best")
# Plot cells in order: light colors first, then dark on top (similar to template's NE then barcoded)
for label, color in boolean_color_map.items():
    mask = BARseq_batch[label]
    ax_cor.scatter(mm_y[mask], mm_z[mask], c=color, s=1, alpha=0.5)

# Add hemisphere labels positioned relative to the actual data range
y_range = ax_cor.get_ylim()
x_range = ax_cor.get_xlim()
label_y = y_range[0] + 0.99 * (y_range[1] - y_range[0])  # Near bottom of plot
left_x = x_range[0] + 0.15 * (x_range[1] - x_range[0])   # 15% from left edge
right_x = x_range[1] - 0.15 * (x_range[1] - x_range[0])  # 15% from right edge

ax_cor.text(left_x, label_y, 'Left Hemisphere', fontsize=16, ha='center', va='center')
ax_cor.text(right_x, label_y, 'Right Hemisphere', fontsize=16, ha='center', va='center')

ax_cor.set_xlabel('M-L axis bregma (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.savefig('/results/plots/BARseq_batch_coronal_plot_bitmap_mesh.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_batch_coronal_plot_bitmap_mesh.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Calculate common axis limits for sagittal plots ---
# Combine mesh and all cell coordinates to get global limits
all_ap_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 0],  # mesh AP
    mm_x                            # all cells AP
])
all_dv_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 1],  # mesh DV
    mm_z                            # all cells DV
])

ap_min, ap_max = all_ap_coords.min(), all_ap_coords.max()
dv_min, dv_max = all_dv_coords.min(), all_dv_coords.max()

# Add margins
ap_margin = 0.05 * (ap_max - ap_min)
dv_margin = 0.05 * (dv_max - dv_min)
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# --- Sagittal Left Hemisphere (AP vs DV) ---
# Create bitmap for left (original mesh)
bitmap_mesh_left, extent_left = create_mesh_bitmap(
    mesh_vertices_bregma_mm[:, 0], mesh_vertices_bregma_mm[:, 1], ap_limits, dv_limits
)

fig_sag_left, ax_sag_left = plt.subplots(figsize=(8, 8))
# Replace scatter with imshow
ax_sag_left.imshow(bitmap_mesh_left, extent=extent_left, origin='lower', aspect='auto', interpolation='nearest')

# Create legend entries for cell types (four categories)
for label, color in boolean_color_map.items():
    ax_sag_left.scatter([], [], c=color, s=10, alpha=0.8, label=label.replace("_", " ").title())  # Format labels nicely

ax_sag_left.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only left hemisphere cells: light colors first, then dark on top
for label, color in boolean_color_map.items():
    mask = BARseq_batch[label] & left_hemisphere_mask
    ax_sag_left.scatter(mm_x[mask], mm_z[mask], c=color, s=1, alpha=0.5)
ax_sag_left.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_left.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_left.set_title('Left Hemisphere', fontsize=16, y=-0.13)
ax_sag_left.set_xlim(ap_limits)
ax_sag_left.set_ylim(dv_limits)
ax_sag_left.invert_yaxis()
ax_sag_left.invert_xaxis()
ax_sag_left.set_aspect('equal')
ax_sag_left.set_frame_on(False)
plt.savefig('/results/plots/BARseq_batch_sagittal_left_plot_bitmap_mesh.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_batch_sagittal_left_plot_bitmap_mesh.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Sagittal Right Hemisphere (AP vs DV) ---
# Create bitmap for right (mirrored mesh)
mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_right[:, 2] *= -1  # Mirror ML for right hemisphere
bitmap_mesh_right, extent_right = create_mesh_bitmap(
    mesh_vertices_bregma_mm_right[:, 0], mesh_vertices_bregma_mm_right[:, 1], ap_limits, dv_limits
)

fig_sag_right, ax_sag_right = plt.subplots(figsize=(8, 8))
# Replace scatter with imshow
ax_sag_right.imshow(bitmap_mesh_right, extent=extent_right, origin='lower', aspect='auto', interpolation='nearest')

# Create legend entries for cell types (four categories)
for label, color in boolean_color_map.items():
    ax_sag_right.scatter([], [], c=color, s=10, alpha=0.8, label=label.replace("_", " ").title())  # Format labels nicely

ax_sag_right.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only right hemisphere cells: light colors first, then dark on top
for label, color in boolean_color_map.items():
    mask = BARseq_batch[label] & right_hemisphere_mask
    ax_sag_right.scatter(mm_x[mask], mm_z[mask], c=color, s=1, alpha=0.5)
ax_sag_right.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_right.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_right.set_title('Right Hemisphere', fontsize=16, y=-0.13)
ax_sag_right.set_xlim(ap_limits)
ax_sag_right.set_ylim(dv_limits)
ax_sag_right.invert_yaxis()
ax_sag_right.invert_xaxis()
ax_sag_right.set_aspect('equal')
ax_sag_right.set_frame_on(False)
plt.savefig('/results/plots/BARseq_batch_sagittal_right_plot_bitmap_mesh.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_batch_sagittal_right_plot_bitmap_mesh.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Set bregma in microns for 25um CCF ---
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

# --- Mesh: subtract bregma ---
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP
mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])  # Mirror ML to left if desired

# --- BARseq_cells: coordinates are already in microns, subtract bregma ---
exa_pts_um = BARseq_TX[["RC", "DV", "ML"]].values  # Use BARseq_TX
exa_pts_bregma_um = exa_pts_um - bregma_um
exa_pts_bregma_um[:, 0] = -exa_pts_bregma_um[:, 0]  # Flip AP

# --- Assign colors using proj_color ---
proj_colors = BARseq_TX["proj_color"].fillna("#cccccc").tolist()

# --- Extract coordinates for plotting (in mm) ---
mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
exa_pts_bregma_mm = exa_pts_bregma_um / 1000.0
mm_x, mm_y, mm_z = exa_pts_bregma_mm[:, 0], exa_pts_bregma_mm[:, 2], exa_pts_bregma_mm[:, 1]  # AP, ML, DV

# --- Create hemisphere masks based on ML coordinates ---
left_hemisphere_mask = mm_y <= 0  # Left hemisphere: ML <= 0
right_hemisphere_mask = mm_y > 0   # Right hemisphere: ML > 0

# --- Define color map for louvain clusters (extract from BARseq_TX to avoid repetition) ---
louvain_color_map = {}
for cluster in sorted(BARseq_TX['louvain_cluster'].unique()):
    color = BARseq_TX.loc[BARseq_TX['louvain_cluster'] == cluster, 'proj_color'].iloc[0]
    louvain_color_map[color] = f"Louvain {cluster}"

# --- Coronal (ML vs DV) ---
fig_cor, ax_cor = plt.subplots(figsize=(12, 8))  # Wider to accommodate both hemispheres
# Combine original and mirrored mesh into one scatter to avoid plotting twice
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1
mesh_combined = np.concatenate([mesh_vertices_bregma_mm[:, [2,1]], mesh_vertices_bregma_mm_mirror[:, [2,1]]])
ax_cor.scatter(mesh_combined[:, 0], mesh_combined[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for louvain clusters
for color, label in louvain_color_map.items():
    ax_cor.scatter([], [], c=color, s=10, alpha=0.8, label=label)

ax_cor.legend(title="Louvain Cluster", loc="best")
# Plot cells grouped by proj_color
grouped = BARseq_TX.groupby("proj_color")
for color, group in grouped:
    ax_cor.scatter(mm_y[group.index], mm_z[group.index], c=color, s=1, alpha=0.5)

# Add hemisphere labels positioned relative to the actual data range
y_range = ax_cor.get_ylim()
x_range = ax_cor.get_xlim()
label_y = y_range[0] + 0.99 * (y_range[1] - y_range[0])  # Near bottom of plot
left_x = x_range[0] + 0.15 * (x_range[1] - x_range[0])   # 15% from left edge
right_x = x_range[1] - 0.15 * (x_range[1] - x_range[0])  # 15% from right edge

ax_cor.text(left_x, label_y, 'Left Hemisphere', fontsize=16, ha='center', va='center')
ax_cor.text(right_x, label_y, 'Right Hemisphere', fontsize=16, ha='center', va='center')

ax_cor.set_xlabel('M-L axis bregma (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.savefig('/results/plots/BARseq_TX_coronal_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_TX_coronal_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Calculate common axis limits for sagittal plots ---
# Combine mesh and all cell coordinates to get global limits
all_ap_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 0],  # mesh AP
    mm_x                            # all cells AP
])
all_dv_coords = np.concatenate([
    mesh_vertices_bregma_mm[:, 1],  # mesh DV
    mm_z                            # all cells DV
])

ap_min, ap_max = all_ap_coords.min(), all_ap_coords.max()
dv_min, dv_max = all_dv_coords.min(), all_dv_coords.max()

# Add margins
ap_margin = 0.05 * (ap_max - ap_min)
dv_margin = 0.05 * (dv_max - dv_min)
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# --- Sagittal Left Hemisphere (AP vs DV) ---
fig_sag_left, ax_sag_left = plt.subplots(figsize=(8, 8))
ax_sag_left.scatter(mesh_vertices_bregma_mm[:, 0], mesh_vertices_bregma_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for louvain clusters
for color, label in louvain_color_map.items():
    ax_sag_left.scatter([], [], c=color, s=10, alpha=0.8, label=label)

ax_sag_left.legend(title="Louvain Cluster", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only left hemisphere cells grouped by proj_color
for color, group in grouped:
    hemisphere_filter = left_hemisphere_mask[group.index]  # Filter group's indices by hemisphere (numpy indexing)
    indices = group.index[hemisphere_filter]
    if len(indices) > 0:
        ax_sag_left.scatter(mm_x[indices], mm_z[indices], c=color, s=1, alpha=0.5)
ax_sag_left.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_left.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_left.set_title('Left Hemisphere', fontsize=16, y=-0.13)
ax_sag_left.set_xlim(ap_limits)
ax_sag_left.set_ylim(dv_limits)
ax_sag_left.invert_yaxis()
ax_sag_left.invert_xaxis()
ax_sag_left.set_aspect('equal')
ax_sag_left.set_frame_on(False)
plt.savefig('/results/plots/BARseq_TX_sagittal_left_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_TX_sagittal_left_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Sagittal Right Hemisphere (AP vs DV) ---
fig_sag_right, ax_sag_right = plt.subplots(figsize=(8, 8))
# Plot right hemisphere mesh (mirrored ML coordinates)
mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_right[:, 2] *= -1  # Mirror ML for right hemisphere
ax_sag_right.scatter(mesh_vertices_bregma_mm_right[:, 0], mesh_vertices_bregma_mm_right[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Create legend entries for louvain clusters
for color, label in louvain_color_map.items():
    ax_sag_right.scatter([], [], c=color, s=10, alpha=0.8, label=label)

ax_sag_right.legend(title="Louvain Cluster", loc="upper left", bbox_to_anchor=(1.02, 1))
# Plot only right hemisphere cells grouped by proj_color
for color, group in grouped:
    hemisphere_filter = right_hemisphere_mask[group.index]  # Filter group's indices by hemisphere (numpy indexing)
    indices = group.index[hemisphere_filter]
    if len(indices) > 0:
        ax_sag_right.scatter(mm_x[indices], mm_z[indices], c=color, s=1, alpha=0.5)
ax_sag_right.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_right.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_right.set_title('Right Hemisphere', fontsize=16, y=-0.13)
ax_sag_right.set_xlim(ap_limits)
ax_sag_right.set_ylim(dv_limits)
ax_sag_right.invert_yaxis()
ax_sag_right.invert_xaxis()
ax_sag_right.set_aspect('equal')
ax_sag_right.set_frame_on(False)
plt.savefig('/results/plots/BARseq_TX_sagittal_right_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_TX_sagittal_right_plot.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Set bregma in microns for 25um CCF ---
bregma_um = np.array([216, 18, 228]) * 25  # [5400, 450, 5700] μm

# --- Mesh: subtract bregma ---
mesh_vertices_bregma_um = mesh_vertices - bregma_um
mesh_vertices_bregma_um[:, 0] = -mesh_vertices_bregma_um[:, 0]  # Flip AP
mesh_vertices_bregma_um[:, 2] = -np.abs(mesh_vertices_bregma_um[:, 2])  # Mirror ML to left if desired

# --- Process BARseq_cells (already in microns) ---
# Sort BARseq_plotting so red cells plot first, blue cells last (for layering)
BARseq_plotting_sorted = BARseq_plotting.sort_values(by="proj_color", key=lambda x: x.map({"#FF0000": 0, "#0000FF": 1})).copy()
bar_pts_um = BARseq_plotting_sorted[["RC", "DV", "ML"]].values  # Already in microns
bar_pts_bregma_um = bar_pts_um - bregma_um
bar_pts_bregma_um[:, 0] = -bar_pts_bregma_um[:, 0]  # Flip AP
bar_colors = BARseq_plotting_sorted["proj_color"].fillna("#cccccc").tolist()
bar_pts_bregma_mm = bar_pts_bregma_um / 1000.0

# --- Process retro_cells (need to scale to microns first) ---
retro_pts_um = retro_cells[["RC", "DV", "ML"]].values * ccf.resolution  # Scale to microns
retro_pts_bregma_um = retro_pts_um - bregma_um
retro_pts_bregma_um[:, 0] = -retro_pts_bregma_um[:, 0]  # Flip AP
retro_pts_bregma_mm = retro_pts_bregma_um / 1000.0

# --- Extract coordinates for plotting (in mm) ---
mesh_vertices_bregma_mm = mesh_vertices_bregma_um / 1000.0
bar_mm_x, bar_mm_y, bar_mm_z = bar_pts_bregma_mm[:, 0], bar_pts_bregma_mm[:, 2], bar_pts_bregma_mm[:, 1]  # AP, ML, DV
retro_mm_x, retro_mm_y, retro_mm_z = retro_pts_bregma_mm[:, 0], retro_pts_bregma_mm[:, 2], retro_pts_bregma_mm[:, 1]  # AP, ML, DV

# --- Create hemisphere masks ---
bar_left_mask = bar_mm_y <= 0
bar_right_mask = bar_mm_y > 0
retro_left_mask = retro_mm_y <= 0
retro_right_mask = retro_mm_y > 0

# --- Coronal (ML vs DV) ---
fig_cor, ax_cor = plt.subplots(figsize=(12, 8))
ax_cor.scatter(mesh_vertices_bregma_mm[:, 2], mesh_vertices_bregma_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)
# Plot mirrored mesh
mesh_vertices_bregma_mm_mirror = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_mirror[:, 2] *= -1
ax_cor.scatter(mesh_vertices_bregma_mm_mirror[:, 2], mesh_vertices_bregma_mm_mirror[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot retro cells FIRST (background layer)
ax_cor.scatter(retro_mm_y, retro_mm_z, c='#228B22', s=1.5, alpha=0.6, zorder=1)
# Plot BARseq cells SECOND (red first, blue on top due to sorting)
ax_cor.scatter(bar_mm_y, bar_mm_z, c=bar_colors, s=2, alpha=0.7, zorder=2)

# Create legend entries
ax_cor.scatter([], [], c="#0000FF", s=10, alpha=0.8, label="Barcoded cells")
ax_cor.scatter([], [], c="#FF0000", s=10, alpha=0.8, label="NE cells")
ax_cor.scatter([], [], c="#228B22", s=8, alpha=0.8, label="Retro cells")
ax_cor.legend(title="Cell Type", loc="best")

# Add hemisphere labels
y_range = ax_cor.get_ylim()
x_range = ax_cor.get_xlim()
label_y = y_range[0] + 0.99 * (y_range[1] - y_range[0])
left_x = x_range[0] + 0.15 * (x_range[1] - x_range[0])
right_x = x_range[1] - 0.15 * (x_range[1] - x_range[0])

ax_cor.text(left_x, label_y, 'Left Hemisphere', fontsize=16, ha='center', va='center')
ax_cor.text(right_x, label_y, 'Right Hemisphere', fontsize=16, ha='center', va='center')

ax_cor.set_xlabel('M-L axis bregma (mm)', fontsize=14)
ax_cor.set_ylabel('D-V axis bregma (mm)', fontsize=14)
ax_cor.invert_yaxis()
ax_cor.set_aspect('equal')
ax_cor.set_frame_on(False)
plt.savefig('/results/plots/BARseq_retro_coronal_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_retro_coronal_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Calculate common axis limits ---
all_ap_coords = np.concatenate([mesh_vertices_bregma_mm[:, 0], bar_mm_x, retro_mm_x])
all_dv_coords = np.concatenate([mesh_vertices_bregma_mm[:, 1], bar_mm_z, retro_mm_z])

ap_min, ap_max = all_ap_coords.min(), all_ap_coords.max()
dv_min, dv_max = all_dv_coords.min(), all_dv_coords.max()

ap_margin = 0.05 * (ap_max - ap_min)
dv_margin = 0.05 * (dv_max - dv_min)
ap_limits = (ap_min - ap_margin, ap_max + ap_margin)
dv_limits = (dv_min - dv_margin, dv_max + dv_margin)

# --- Sagittal Left Hemisphere ---
fig_sag_left, ax_sag_left = plt.subplots(figsize=(8, 8))
ax_sag_left.scatter(mesh_vertices_bregma_mm[:, 0], mesh_vertices_bregma_mm[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot retro cells FIRST (background layer)
ax_sag_left.scatter(retro_mm_x[retro_left_mask], retro_mm_z[retro_left_mask], c='#228B22', s=1.5, alpha=0.6, zorder=1)
# Plot BARseq cells SECOND (red first, blue on top due to sorting)
left_bar_colors = [bar_colors[i] for i in range(len(bar_colors)) if bar_left_mask[i]]
ax_sag_left.scatter(bar_mm_x[bar_left_mask], bar_mm_z[bar_left_mask], c=left_bar_colors, s=2, alpha=0.7, zorder=2)

# Legend
ax_sag_left.scatter([], [], c="#0000FF", s=10, alpha=0.8, label="Barcoded cells")
ax_sag_left.scatter([], [], c="#FF0000", s=10, alpha=0.8, label="NE cells")
ax_sag_left.scatter([], [], c="#228B22", s=8, alpha=0.8, label="Retro cells")
ax_sag_left.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))

ax_sag_left.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_left.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_left.set_title('Left Hemisphere', fontsize=16, y=-0.13)
ax_sag_left.set_xlim(ap_limits)
ax_sag_left.set_ylim(dv_limits)
ax_sag_left.invert_yaxis()
ax_sag_left.invert_xaxis()
ax_sag_left.set_aspect('equal')
ax_sag_left.set_frame_on(False)
plt.savefig('/results/plots/BARseq_retro_sagittal_left_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_retro_sagittal_left_plot.svg', format='svg', bbox_inches='tight')
plt.show()

# --- Sagittal Right Hemisphere ---
fig_sag_right, ax_sag_right = plt.subplots(figsize=(8, 8))
mesh_vertices_bregma_mm_right = mesh_vertices_bregma_mm.copy()
mesh_vertices_bregma_mm_right[:, 2] *= -1
ax_sag_right.scatter(mesh_vertices_bregma_mm_right[:, 0], mesh_vertices_bregma_mm_right[:, 1], color='gray', s=dot_size, alpha=mesh_alpha)

# Plot retro cells FIRST (background layer)
ax_sag_right.scatter(retro_mm_x[retro_right_mask], retro_mm_z[retro_right_mask], c='#228B22', s=1.5, alpha=0.6, zorder=1)
# Plot BARseq cells SECOND (red first, blue on top due to sorting)
right_bar_colors = [bar_colors[i] for i in range(len(bar_colors)) if bar_right_mask[i]]
ax_sag_right.scatter(bar_mm_x[bar_right_mask], bar_mm_z[bar_right_mask], c=right_bar_colors, s=2, alpha=0.7, zorder=2)

# Legend
ax_sag_right.scatter([], [], c="#0000FF", s=10, alpha=0.8, label="Barcoded cells")
ax_sag_right.scatter([], [], c="#FF0000", s=10, alpha=0.8, label="NE cells")
ax_sag_right.scatter([], [], c="#228B22", s=8, alpha=0.8, label="Retro cells")
ax_sag_right.legend(title="Cell Type", loc="upper left", bbox_to_anchor=(1.02, 1))

ax_sag_right.set_xlabel('A-P bregma axis (mm)', fontsize=14)
ax_sag_right.set_ylabel('D-V bregma axis (mm)', fontsize=14)
ax_sag_right.set_title('Right Hemisphere', fontsize=16, y=-0.13)
ax_sag_right.set_xlim(ap_limits)
ax_sag_right.set_ylim(dv_limits)
ax_sag_right.invert_yaxis()
ax_sag_right.invert_xaxis()
ax_sag_right.set_aspect('equal')
ax_sag_right.set_frame_on(False)
plt.savefig('/results/plots/BARseq_retro_sagittal_right_plot.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('/results/plots/BARseq_retro_sagittal_right_plot.svg', format='svg', bbox_inches='tight')
plt.show()